# Statistische Validierung der Pokémon- und Recommender-Grundlagen

Dieses Notebook ergänzt die deskriptive EDA um inferenzstatistische Verfahren. Es untersucht:

1. Pearson- und Spearman-Zusammenhänge zwischen den sechs Basiswerten,
2. den Unterschied zwischen finalen und nicht finalen Entwicklungen,
3. den rohen Unterschied zwischen Einzel- und Doppeltypen,
4. denselben Typenvergleich getrennt nach Evolutionsstufe und Finalstatus.

Neben p-Werten berichten wir Effektgrößen und Bootstrap-Konfidenzintervalle. Der Datensatz enthält nahezu die vollständige betrachtete Pokémon-Population und keine Zufallsstichprobe. Die Tests quantifizieren daher Muster des Datensatzes; sie belegen keine kausalen Designentscheidungen.

In [1]:
from dataclasses import asdict
from html import escape
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, display

from pokemon_team_advisor.statistical_analysis import (
    CorrelationMethod,
    bootstrap_mean_difference,
    calculate_correlation,
    compare_independent_groups,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

RANDOM_SEED = 42
STAT_COLUMNS = [
    "hp",
    "attack",
    "defense",
    "special_attack",
    "special_defense",
    "speed",
]

## 1. Daten laden und Voraussetzungen prüfen

In [8]:
candidate_paths = [
    Path("data/processed/pokemon.csv"),
    Path("../data/processed/pokemon.csv"),
]
data_path = next((path for path in candidate_paths if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError("data/processed/pokemon.csv wurde nicht gefunden.")

pokemon = pd.read_csv(data_path)
required_columns = {
    "id",
    "name",
    "type_1",
    "type_2",
    "base_stat_total",
    "evolution_stage",
    "is_final_evolution",
    *STAT_COLUMNS,
}
missing_columns = required_columns.difference(pokemon.columns)
if missing_columns:
    raise ValueError(f"Fehlende Spalten: {sorted(missing_columns)}")

if pokemon[list(required_columns)].drop(columns="type_2").isna().any().any():
    raise ValueError("Benötigte Analysewerte enthalten fehlende Einträge.")
if pokemon["id"].duplicated().any():
    raise ValueError("Pokémon-IDs müssen eindeutig sein.")
if not pokemon[STAT_COLUMNS].apply(np.isfinite).all().all():
    raise ValueError("Basiswerte müssen endlich sein.")
if not pokemon["base_stat_total"].eq(pokemon[STAT_COLUMNS].sum(axis=1)).all():
    raise ValueError("Mindestens ein Gesamtbasiswert ist inkonsistent.")

pokemon = pokemon.assign(is_dual_type=pokemon["type_2"].notna())

print(f"Geladene Datei: {data_path.as_posix()}")
print(f"Pokémon: {len(pokemon):,}".replace(",", "."))
display(
    pokemon[
        [
            "id",
            "name",
            "type_1",
            "type_2",
            "base_stat_total",
            "evolution_stage",
            "is_final_evolution",
        ]
    ].head()
)

Geladene Datei: ../data/processed/pokemon.csv
Pokémon: 1.025


,id,name,type_1,type_2,base_stat_total,evolution_stage,is_final_evolution
0,1,bulbasaur,grass,poison,318,0,False
1,2,ivysaur,grass,poison,405,1,False
2,3,venusaur,grass,poison,525,2,True
3,4,charmander,fire,NaN,309,0,False
4,5,charmeleon,fire,NaN,405,1,False


## 2. Korrelationen der Basiswerte

Pearson misst lineare Zusammenhänge. Spearman arbeitet mit Rangplätzen und erkennt auch monotone, nicht lineare Beziehungen. Für alle 30 Tests wird zusätzlich eine konservative Bonferroni-Korrektur berechnet. Da die sechs Werte gemeinsam das Profil eines Pokémon bilden, interpretieren wir die Koeffizienten stärker als die p-Werte.

In [3]:
correlation_rows: list[dict[str, object]] = []
for column_x, column_y in combinations(STAT_COLUMNS, 2):
    for method in CorrelationMethod:
        result = calculate_correlation(
            pokemon[column_x],
            pokemon[column_y],
            method=method,
        )
        correlation_rows.append(
            {
                "variable_x": column_x,
                "variable_y": column_y,
                **asdict(result),
            }
        )

correlations = pd.DataFrame(correlation_rows)
correlations["p_value_bonferroni"] = np.minimum(
    correlations["p_value"] * len(correlations),
    1.0,
)
correlations["absolute_coefficient"] = correlations["coefficient"].abs()

display(
    correlations.sort_values(
        ["absolute_coefficient", "method"],
        ascending=[False, True],
    ).reset_index(drop=True)
)

,variable_x,variable_y,method,coefficient,p_value,sample_size,p_value_bonferroni,absolute_coefficient
0,hp,attack,spearman,0.5948,0.0000,1025,0.0000,0.5948
1,defense,special_defense,spearman,0.5768,0.0000,1025,0.0000,0.5768
2,special_attack,special_defense,spearman,0.5569,0.0000,1025,0.0000,0.5569
3,attack,defense,spearman,0.5260,0.0000,1025,0.0000,0.5260
4,hp,special_defense,spearman,0.5039,0.0000,1025,0.0000,0.5039
5,defense,special_defense,pearson,0.5033,0.0000,1025,0.0000,0.5033
6,special_attack,special_defense,pearson,0.4930,0.0000,1025,0.0000,0.4930
7,hp,defense,spearman,0.4894,0.0000,1025,0.0000,0.4894
8,hp,attack,pearson,0.4761,0.0000,1025,0.0000,0.4761
9,hp,special_attack,spearman,0.4674,0.0000,1025,0.0000,0.4674


In [4]:
pearson_matrix = pokemon[STAT_COLUMNS].corr(method="pearson")


def correlation_heatmap_html(matrix: pd.DataFrame) -> str:
    header_cells = "".join(
        f"<th style='padding:8px'>{escape(str(column))}</th>" for column in matrix.columns
    )
    body_rows: list[str] = []
    for row_name, row in matrix.iterrows():
        value_cells: list[str] = []
        for raw_value in row:
            value = float(raw_value)
            red, green, blue = (180, 45, 45) if value >= 0.0 else (45, 90, 180)
            alpha = 0.08 + 0.72 * min(abs(value), 1.0)
            value_cells.append(
                "<td style='padding:8px;text-align:center;"
                f"background:rgba({red},{green},{blue},{alpha:.3f})'>"
                f"{value:.2f}</td>"
            )
        body_rows.append(
            f"<tr><th style='padding:8px;text-align:left'>{escape(str(row_name))}</th>"
            + "".join(value_cells)
            + "</tr>"
        )

    return (
        "<table style='border-collapse:collapse'>"
        "<caption style='font-weight:600;margin-bottom:8px'>"
        "Pearson-Korrelationen der sechs Basiswerte</caption>"
        f"<thead><tr><th></th>{header_cells}</tr></thead>"
        f"<tbody>{''.join(body_rows)}</tbody></table>"
    )


display(HTML(correlation_heatmap_html(pearson_matrix)))

,hp,attack,defense,special_attack,special_defense,speed
hp,1.00,0.48,0.30,0.36,0.37,0.18
attack,0.48,1.00,0.47,0.28,0.23,0.35
defense,0.30,0.47,1.00,0.21,0.50,0.01
special_attack,0.36,0.28,0.21,1.00,0.49,0.42
special_defense,0.37,0.23,0.50,0.49,1.00,0.21
speed,0.18,0.35,0.01,0.42,0.21,1.00


## 3. Finale gegen nicht finale Entwicklungen

Die vorab festgelegte Hypothese lautet: Finale Entwicklungen besitzen im Mittel einen höheren Gesamtbasiswert. Wir verwenden den Welch-t-Test, da er keine gleichen Varianzen voraussetzt. Der Levene-Test wird nur als Varianzdiagnose berichtet. Hedges’ $g$ beschreibt die standardisierte Effektgröße; das Bootstrap-Intervall quantifiziert die Unsicherheit der Mittelwertdifferenz.

In [5]:
final_totals = pokemon.loc[pokemon["is_final_evolution"], "base_stat_total"].astype(float)
non_final_totals = pokemon.loc[~pokemon["is_final_evolution"], "base_stat_total"].astype(float)

final_comparison = compare_independent_groups(final_totals, non_final_totals)
final_interval = bootstrap_mean_difference(
    final_totals,
    non_final_totals,
    confidence_level=0.95,
    resamples=5_000,
    random_seed=RANDOM_SEED,
)

display(
    pd.DataFrame(
        [
            {
                "comparison": "final minus non-final",
                **asdict(final_comparison),
                "ci_lower": final_interval.lower_bound,
                "ci_upper": final_interval.upper_bound,
            }
        ]
    )
)

,comparison,sample_size_a,sample_size_b,mean_a,mean_b,mean_difference,levene_statistic,levene_p_value,welch_t_statistic,welch_p_value,hedges_g,ci_lower,ci_upper
0,final minus non-final,568,457,509.3063,326.2429,183.0634,1.8634,0.1725,43.6965,0.0000,2.7473,174.9161,191.4766


## 4. Roher Vergleich von Einzel- und Doppeltypen

Dieser Vergleich ist bewusst zunächst unkontrolliert. Ein signifikanter Unterschied darf nicht direkt dem zweiten Typ zugeschrieben werden: Doppeltypen und Einzeltypen können sich in Evolutionsstufe, Finalstatus und Zusammensetzung unterscheiden.

In [6]:
dual_type_totals = pokemon.loc[pokemon["is_dual_type"], "base_stat_total"].astype(float)
single_type_totals = pokemon.loc[~pokemon["is_dual_type"], "base_stat_total"].astype(float)

raw_type_comparison = compare_independent_groups(
    dual_type_totals,
    single_type_totals,
)
raw_type_interval = bootstrap_mean_difference(
    dual_type_totals,
    single_type_totals,
    confidence_level=0.95,
    resamples=5_000,
    random_seed=RANDOM_SEED,
)

display(
    pd.DataFrame(
        [
            {
                "comparison": "dual minus single",
                **asdict(raw_type_comparison),
                "ci_lower": raw_type_interval.lower_bound,
                "ci_upper": raw_type_interval.upper_bound,
            }
        ]
    )
)

,comparison,sample_size_a,sample_size_b,mean_a,mean_b,mean_difference,levene_statistic,levene_p_value,welch_t_statistic,welch_p_value,hedges_g,ci_lower,ci_upper
0,dual minus single,526,499,451.0285,403.0822,47.9464,2.7103,0.1000,6.9574,0.0000,0.4346,34.2620,61.7132


## 5. Typenvergleich innerhalb vergleichbarer Evolutionsgruppen

Nun vergleichen wir nur Pokémon mit derselben Evolutionsstufe und demselben Finalstatus. Damit kontrollieren wir zwei wichtige Confounder. Gruppen mit weniger als zwei Beobachtungen auf einer Seite werden ausgeschlossen. Da mehrere Strata getestet werden, verwenden wir für die Entscheidung eine Bonferroni-korrigierte Signifikanzgrenze.

In [7]:
stratified_rows: list[dict[str, object]] = []
group_columns = ["evolution_stage", "is_final_evolution"]

for group_index, (group_key, group) in enumerate(pokemon.groupby(group_columns, observed=True)):
    evolution_stage, is_final_evolution = group_key
    dual_values = group.loc[group["is_dual_type"], "base_stat_total"].astype(float)
    single_values = group.loc[~group["is_dual_type"], "base_stat_total"].astype(float)
    if len(dual_values) < 2 or len(single_values) < 2:
        continue

    comparison = compare_independent_groups(dual_values, single_values)
    interval = bootstrap_mean_difference(
        dual_values,
        single_values,
        confidence_level=0.95,
        resamples=2_000,
        random_seed=RANDOM_SEED + group_index,
    )
    stratified_rows.append(
        {
            "evolution_stage": int(evolution_stage),
            "is_final_evolution": bool(is_final_evolution),
            **asdict(comparison),
            "ci_lower": interval.lower_bound,
            "ci_upper": interval.upper_bound,
        }
    )

stratified_comparisons = pd.DataFrame(stratified_rows)
if stratified_comparisons.empty:
    print("Keine Strata besitzen auf beiden Seiten mindestens zwei Beobachtungen.")
else:
    corrected_alpha = 0.05 / len(stratified_comparisons)
    stratified_comparisons["bonferroni_alpha"] = corrected_alpha
    stratified_comparisons["significant_after_correction"] = (
        stratified_comparisons["welch_p_value"] < corrected_alpha
    )
    display(stratified_comparisons)

,evolution_stage,is_final_evolution,sample_size_a,sample_size_b,mean_a,mean_b,mean_difference,levene_statistic,levene_p_value,welch_t_statistic,welch_p_value,hedges_g,ci_lower,ci_upper,bonferroni_alpha,significant_after_correction
0,0,False,134,206,310.0597,300.2282,9.8315,0.4655,0.4955,1.6286,0.1046,0.1826,-1.8570,21.7368,0.0100,False
1,0,True,124,77,550.0081,509.4935,40.5146,8.9586,0.0031,3.0341,0.0029,0.4735,14.9565,66.3144,0.0100,True
2,1,False,54,63,387.5000,393.2222,-5.7222,0.0921,0.7621,-0.5201,0.6040,-0.0950,-27.6879,16.6190,0.0100,False
3,1,True,131,115,484.5038,481.6000,2.9038,0.0055,0.9408,0.5792,0.5630,0.0733,-6.8463,12.6537,0.0100,False
4,2,True,83,38,519.2410,523.7632,-4.5222,4.2428,0.0416,-0.5702,0.5697,-0.0942,-20.9656,10.1883,0.0100,False


## 6. Statistische Erkenntnisse und Schlussfolgerungen

### Ziel der Analyse

Die statistische Analyse ergänzt die deskriptive EDA um Korrelationsanalysen, Gruppenvergleiche, Effektgrößen und Bootstrap-Konfidenzintervalle. Untersucht wurden insbesondere:

1. Zusammenhänge zwischen den sechs Basiswerten,
2. Unterschiede zwischen finalen und nicht finalen Entwicklungen,
3. der rohe Unterschied zwischen Einzel- und Doppeltypen,
4. der Typenunterschied innerhalb vergleichbarer Evolutionsgruppen.

Neben p-Werten werden Hedges’ \(g\) und 95-%-Bootstrap-Konfidenzintervalle berichtet. Dadurch lässt sich nicht nur beurteilen, ob ein Unterschied statistisch auffällig ist, sondern auch, wie groß und stabil er ausfällt.

---

### Korrelationsanalyse

Für alle 15 möglichen Paare der sechs Basiswerte wurden sowohl Pearson- als auch Spearman-Korrelationen berechnet.

* Pearson beschreibt lineare Zusammenhänge.
* Spearman beschreibt monotone Zusammenhänge auf Basis der Rangfolge.
* Insgesamt entstehen 30 Korrelationsprüfungen.
* Die p-Werte wurden deshalb mit einer Bonferroni-Korrektur gegen zufällige Mehrfachtreffer abgesichert.
* Der Gesamtbasiswert wurde bewusst nicht in die Korrelationsmatrix aufgenommen, da er exakt aus der Summe der sechs Einzelwerte besteht. Eine Korrelation mit seinen Bestandteilen wäre daher teilweise konstruktionsbedingt.

Die Korrelationsmatrix beschreibt typische Werteprofile. Positive Korrelationen zeigen, welche Basiswerte häufig gemeinsam hoch oder niedrig ausfallen. Schwache oder negative Korrelationen können auf Spezialisierung und statistische Zielkonflikte zwischen verschiedenen Rollen hinweisen.

Die Zusammenhänge dürfen nicht kausal interpretiert werden: Eine hohe Verteidigung verursacht beispielsweise keinen niedrigen oder hohen Geschwindigkeitswert.

---

## Finale und nicht finale Entwicklungen

Verglichen wurden `568` finale und `457` nicht finale Pokémon.

| Kennzahl                               |         Ergebnis |
| -------------------------------------- | ---------------: |
| Mittelwert finaler Entwicklungen       |           509,31 |
| Mittelwert nicht finaler Entwicklungen |           326,24 |
| Mittelwertdifferenz                    |          +183,06 |
| Levene-Statistik                       |             1,86 |
| Levene-p-Wert                          |           0,1725 |
| Welch-t-Statistik                      |            43,70 |
| Welch-p-Wert                           |         < 0,0001 |
| Hedges’ \(g\)                          |             2,75 |
| 95-%-Bootstrap-Intervall               | [174,92; 191,48] |

Der Levene-Test liefert mit \(p=0{,}1725\) keinen deutlichen Hinweis auf unterschiedliche Varianzen. Unabhängig davon wurde der robustere Welch-t-Test verwendet.

Finale Entwicklungen besitzen im Mittel ungefähr `183` zusätzliche Gesamtbasispunkte. Das Bootstrap-Intervall liegt vollständig oberhalb von null und bestätigt eine stabile positive Differenz.

Hedges’ \(g=2{,}75\) beschreibt einen außergewöhnlich großen standardisierten Unterschied. Der Unterschied ist damit nicht nur statistisch auffällig, sondern auch praktisch sehr bedeutend.

### Konsequenz für den Recommender

Die Entscheidung, standardmäßig nur finale Entwicklungen zu empfehlen, wird durch die Analyse klar unterstützt. Nicht finale Pokémon können ein deutliches Rollenprofil besitzen, sind im Durchschnitt aber wesentlich schwächer.

Die Option `include_non_final=True` bleibt trotzdem sinnvoll für Spezialfälle, persönliche Präferenzen oder besondere Spielregeln.

---

## Roher Vergleich von Doppel- und Einzeltypen

Verglichen wurden `526` Doppeltypen und `499` Einzeltypen.

| Kennzahl                 |       Ergebnis |
| ------------------------ | -------------: |
| Mittelwert Doppeltypen   |         451,03 |
| Mittelwert Einzeltypen   |         403,08 |
| Mittelwertdifferenz      |         +47,95 |
| Levene-Statistik         |           2,71 |
| Levene-p-Wert            |         0,1000 |
| Welch-t-Statistik        |           6,96 |
| Welch-p-Wert             |       < 0,0001 |
| Hedges’ \(g\)            |           0,43 |
| 95-%-Bootstrap-Intervall | [34,26; 61,71] |

Im unkontrollierten Vergleich besitzen Doppeltypen durchschnittlich etwa `47,95` zusätzliche Gesamtbasispunkte.

Das Bootstrap-Intervall liegt vollständig oberhalb von null. Hedges’ \(g=0{,}43\) entspricht einem kleinen bis mittleren standardisierten Unterschied.

Dieser rohe Vergleich reicht jedoch nicht für die Schlussfolgerung aus, dass ein zweiter Typ ein Pokémon grundsätzlich stärker macht. Doppel- und Einzeltypen unterscheiden sich auch hinsichtlich Evolutionsstufe, Finalstatus und Gruppenzusammensetzung.

---

## Kontrollierter Typenvergleich

Zur Kontrolle wichtiger Confounder wurden Einzel- und Doppeltypen nur innerhalb derselben Evolutionsstufe und desselben Finalstatus verglichen.

Da fünf Gruppen getestet wurden, gilt nach Bonferroni-Korrektur:

$$
\alpha_{\text{korrigiert}}=\frac{0{,}05}{5}=0{,}01
$$

| Evolutionsgruppe     | Doppeltypen | Einzeltypen | Differenz | Welch-\(p\) | Hedges’ \(g\) | 95-%-Bootstrap-Intervall | Korrigiert signifikant |
| -------------------- | ----------: | ----------: | --------: | ----------: | ------------: | -----------------------: | ---------------------- |
| Stufe 0, nicht final |         134 |         206 |     +9,83 |      0,1046 |          0,18 |           [−1,86; 21,74] | Nein                   |
| Stufe 0, final       |         124 |          77 |    +40,51 |      0,0029 |          0,47 |           [14,96; 66,31] | Ja                     |
| Stufe 1, nicht final |          54 |          63 |     −5,72 |      0,6040 |         −0,10 |          [−27,69; 16,62] | Nein                   |
| Stufe 1, final       |         131 |         115 |     +2,90 |      0,5630 |          0,07 |           [−6,85; 12,65] | Nein                   |
| Stufe 2, final       |          83 |          38 |     −4,52 |      0,5697 |         −0,09 |          [−20,97; 10,19] | Nein                   |

### Interpretation

In vier von fünf kontrollierten Gruppen sind die Unterschiede klein:

* Die Effektgrößen liegen zwischen ungefähr \(-0{,}10\) und \(0{,}18\).
* Alle zugehörigen Bootstrap-Intervalle enthalten null.
* Keine dieser vier Gruppen überschreitet die Bonferroni-korrigierte Signifikanzgrenze.

Nur bei finalen Pokémon auf Evolutionsstufe 0 bleibt ein deutlicher Unterschied bestehen:

* Doppeltypen besitzen durchschnittlich `40,51` zusätzliche Punkte.
* Hedges’ \(g=0{,}47\) beschreibt einen kleinen bis mittleren Effekt.
* Das Bootstrap-Intervall `[14,96; 66,31]` liegt vollständig oberhalb von null.
* Der Welch-p-Wert von `0,0029` bleibt auch nach der Bonferroni-Korrektur signifikant.

In dieser Gruppe befinden sich eigenständige finale Pokémon, Legendäre und andere besonders starke Formen. Der Unterschied kann deshalb nicht automatisch dem zweiten Typ zugeschrieben werden.

---

## Varianzdiagnostik

Der Levene-Test zeigt in zwei kontrollierten Gruppen Hinweise auf unterschiedliche Varianzen:

| Evolutionsgruppe | Levene-\(p\) | Interpretation                |
| ---------------- | -----------: | ----------------------------- |
| Stufe 0, final   |       0,0031 | deutlicher Varianzunterschied |
| Stufe 2, final   |       0,0416 | möglicher Varianzunterschied  |

Das ändert die Teststrategie nicht. Für alle Gruppen wurde von Anfang an der Welch-t-Test eingesetzt, da er keine gleichen Varianzen voraussetzt.

Der Levene-Test wird damit als Diagnose berichtet und nicht als automatischer Schalter zwischen Student- und Welch-t-Test verwendet.

---

## Vergleich mit dem unkontrollierten Ergebnis

Der rohe Doppeltypenunterschied beträgt:

$$
+47{,}95
$$

Nach Kontrolle von Evolutionsstufe und Finalstatus liegen vier der fünf Unterschiede dagegen nur zwischen:

$$
-5{,}72 \quad \text{und} \quad +9{,}83
$$

Das zeigt einen starken Confounding-Effekt. Ein großer Teil des ursprünglichen Doppeltypenunterschieds entsteht durch die unterschiedliche Zusammensetzung der Gruppen.

Der zweite Typ ist daher kein allgemeiner Indikator für höhere Basiswerte.

---

## Konsequenzen für das Recommender-Scoring

Aus der statistischen Analyse folgen vier konkrete Modellentscheidungen:

1. **Finale Entwicklungen bleiben der Standardfilter.**
   Der sehr große Unterschied von Hedges’ \(g=2{,}75\) unterstützt den Ausschluss nicht finaler Entwicklungen aus den normalen Empfehlungen.

2. **Doppeltypen erhalten keinen pauschalen Stärkebonus.**
   Der rohe Vorteil verschwindet in den meisten vergleichbaren Evolutionsgruppen nahezu vollständig.

3. **Die konkrete defensive Ergänzung bleibt entscheidend.**
   Ein zweiter Typ ist nur dann vorteilhaft, wenn er relevante Teamschwächen reduziert, Resistenzen ergänzt oder Immunitäten bereitstellt.

4. **Rollenpassung und Stärke bleiben getrennte Komponenten.**
   Ein schwaches Pokémon kann ein klares Profil besitzen, soll dadurch aber nicht automatisch mit einem deutlich stärkeren Kandidaten gleichgestellt werden.

Die aktuelle Recommender-Struktur entspricht diesen Erkenntnissen:

$$
\text{Gesamtscore}
=
0{,}50 \cdot \text{Defensive Ergänzung}
+
0{,}30 \cdot \text{Rollenpassung}
+
0{,}20 \cdot \text{Stärke}
$$

Die Gewichte sind derzeit fachlich begründete Modellparameter. Die statistischen Gruppenvergleiche beweisen nicht, dass genau diese Gewichtung optimal ist. Das muss später separat durch Offline-Evaluation und einen echten A/B-Test untersucht werden.

---

## Methodische Grenzen

Die Ergebnisse müssen unter mehreren Einschränkungen interpretiert werden:

* Der Datensatz enthält nahezu die vollständige betrachtete Pokémon-Population und keine klassische Zufallsstichprobe.
* Pokémon derselben Evolutionsfamilie sind nicht vollständig unabhängig.
* Regionale Formen und verzweigte Entwicklungen können ähnliche Eigenschaften teilen.
* Die Gruppen wurden nicht experimentell randomisiert.
* Statistische Signifikanz beweist keine Kausalität.
* Ein kleiner p-Wert sagt nichts über die praktische Größe eines Effekts aus.
* Schwellenwerte für kleine, mittlere oder große Effektgrößen sind nur Orientierungshilfen.
* Die Analyse verwendet Basiswerte, Typen und Evolutionsinformationen, aber noch keine Attacken, Fähigkeiten, Items oder formatspezifischen Regeln.

Deshalb werden Effektgrößen, Konfidenzintervalle und fachliche Plausibilität gemeinsam betrachtet.

---

## Vorbereitung eines echten A/B-Tests

Die bisherige Analyse ist kein A/B-Test. Ein echter A/B-Test benötigt reale Nutzer, die zufällig unterschiedlichen Recommender-Varianten zugewiesen werden.

Eine mögliche spätere Untersuchung wäre:

* **Kontrollgruppe A:** aktuelle Gewichtung `0,50 / 0,30 / 0,20`
* **Variante B:** alternative Gewichtung mit stärkerer Rollen- oder Stärkepriorisierung
* **Primäre Zielgröße:** Anteil der angezeigten Empfehlungen, die ins Team übernommen werden
* **Sekundäre Zielgrößen:** Interaktionen mit Erklärungen, Änderung der Empfehlung und Abschluss eines Teams
* **Zuweisung:** zufällig, aber pro Nutzer oder Sitzung stabil
* **Auswertung:** vorab festgelegte Hypothese, Mindeststichprobe, Testdauer, Effektgröße und Konfidenzintervall

Eine zufällige Aufteilung der Pokémon-Datensätze wäre dagegen kein A/B-Test, weil dabei keine Nutzer unterschiedliche Produktvarianten erleben.

---

## Gesamtfazit

Die statistische Analyse bestätigt zwei zentrale Designentscheidungen des Projekts:

* Finale Entwicklungen sind im Durchschnitt erheblich stärker als nicht finale Entwicklungen.
* Doppeltypen sind nicht grundsätzlich stärker, sobald wichtige Unterschiede im Evolutionsstand kontrolliert werden.

Damit sollte der Recommender weder ein klares Rollenprofil mit absoluter Stärke verwechseln noch einen zweiten Typ pauschal belohnen. Gute Empfehlungen entstehen durch die Kombination aus konkreter defensiver Ergänzung, noch offenen Teamrollen und ausreichender absoluter Stärke.

Die statistischen Ergebnisse unterstützen somit die aktuelle Modellarchitektur, ohne kausale Aussagen zu beanspruchen.
